# 2e, Koszul bracket anti-symmetry: $[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}$

**Problem (e).** The Koszul bracket on $T^*M$ is anti-symmetric:

$$
[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}.
$$

The definition of the Koszul bracket:

$$
[\alpha, \beta]_K \;=\; \mathcal{L}_{\pi^\sharp\alpha}\beta \;-\; \mathcal{L}_{\pi^\sharp\beta}\alpha \;-\; d\langle \pi^\sharp\alpha, \beta \rangle.
$$

When we sum the two Koszul definitions, the Lie-derivative terms cancel against their counterparts; only $-d\bigl(\langle X_\alpha,\beta\rangle + \langle X_\beta,\alpha\rangle\bigr)$ remains. Since $\pi$ is anti-symmetric, this pairing sum is zero.

## Strategy, why noticeably easier than 2d

2d demanded **$C^\infty$-module algebra**: sharp tensoriality, $\mathcal{L}_{fX}$ expansion, $df\wedge\iota$. **None** of those appear in 2e, anti-symmetry is a purely algebraic cancellation:

$$
[\alpha,\beta]_K + [\beta,\alpha]_K \;=\; \bigl(L_{X_\alpha}\beta - L_{X_\beta}\alpha\bigr) + \bigl(L_{X_\beta}\alpha - L_{X_\alpha}\beta\bigr) - d\langle X_\alpha,\beta\rangle - d\langle X_\beta,\alpha\rangle.
$$

The Lie-derivative part **directly** cancels, no expansion needed. What remains:

$$
-d\bigl(\langle X_\alpha,\beta\rangle + \langle X_\beta,\alpha\rangle\bigr) \;=\; -d(\pi(\alpha,\beta) + \pi(\beta,\alpha)) \;=\; 0.
$$

**Single geometric input:** $\pi$ anti-symmetry, at the pairing level: $\langle X_\alpha,\beta\rangle = -\langle X_\beta,\alpha\rangle$.

**Mode discipline.** Both vector fields are **flow-mode**. If they were Cartan-mode, $L_{X_\alpha}\beta \to (d\iota_{X_\alpha} + \iota_{X_\alpha}d)\beta$ would open, $d(\beta)$ from a generic 1-form would be irreducible, and the chain would stall. They need to **stay opaque** in order to cancel, `flow` provides exactly that.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.calculus.pairing import pairing, Pairing
from jacopy.core.expr import Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

## 1. Setup

- $\alpha, \beta$, generic 1-forms.
- $X_\alpha = \pi^\sharp(\alpha)$, $X_\beta = \pi^\sharp(\beta)$, named `Derivation`s (keeping sharp intrinsically as a tensor depends on Phase 12 #6; in this pass we directly model the resulting vector fields).
- $\mathcal{L}_{X_\alpha}, \mathcal{L}_{X_\beta}$ **flow-mode**, so the Lie-derivative terms stay opaque and cancel each other.

In [2]:
reg = PropertyRegistry()

alpha = Symbol("α")
beta = Symbol("β")
reg.declare(alpha, Graded(degree=1))
reg.declare(beta, Graded(degree=1))

X_a = Derivation("X_α", degree=0)
X_b = Derivation("X_β", degree=0)
L_Xa = lie_derivative(X_a, definition="flow")
L_Xb = lie_derivative(X_b, definition="flow")

koszul_ab = Symbol("[α,β]_K")
koszul_ba = Symbol("[β,α]_K")
reg.declare(koszul_ab, Graded(degree=1))
reg.declare(koszul_ba, Graded(degree=1))

print(f"α, β       : {alpha} {beta}")
print(f"X_α, X_β   : {X_a} {X_b}  (both flow-mode)")
print(f"[α,β]_K    : {koszul_ab}")
print(f"[β,α]_K    : {koszul_ba}")

α, β       : α β
X_α, X_β   : X_α X_β  (both flow-mode)
[α,β]_K    : [α,β]_K
[β,α]_K    : [β,α]_K


## 2. Axioms (3 of them)

**A-K1, `[α, β]_K` defining.** Standard Koszul definition:

$$
[\alpha, \beta]_K \;\to\; \mathcal{L}_{X_\alpha}\beta - \mathcal{L}_{X_\beta}\alpha - d\langle X_\alpha, \beta\rangle.
$$

**A-K2, `[β, α]_K` defining.** Same formula, with the roles swapped:

$$
[\beta, \alpha]_K \;\to\; \mathcal{L}_{X_\beta}\alpha - \mathcal{L}_{X_\alpha}\beta - d\langle X_\beta, \alpha\rangle.
$$

**A-π-pairing-antisym, $\pi$ anti-symmetry at the pairing level.**

$$
\langle X_\alpha, \beta\rangle = -\langle X_\beta, \alpha\rangle.
$$

Because $\langle\pi^\sharp\alpha,\beta\rangle = \beta(\pi^\sharp\alpha) = \pi(\alpha,\beta) = -\pi(\beta,\alpha) = -\langle\pi^\sharp\beta,\alpha\rangle$. The single **genuine geometric input**, drops from axiom to theorem when Phase 12 infrastructure #11 (`AntiSymmetric` registry property + bivector eval rewrite) lands.

In [3]:
class KoszulDef_ab(Definition):
    """A-K1: [α, β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩."""
    name = "[α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩"
    def matches(self, expr): return expr == koszul_ab
    def rewrite(self, expr):
        return Sum(
            Act(L_Xa, beta),
            Neg(Act(L_Xb, alpha)),
            Neg(Act(d, pairing(X_a, beta))),
        )


class KoszulDef_ba(Definition):
    """A-K2: [β, α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩."""
    name = "[β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩"
    def matches(self, expr): return expr == koszul_ba
    def rewrite(self, expr):
        return Sum(
            Act(L_Xb, alpha),
            Neg(Act(L_Xa, beta)),
            Neg(Act(d, pairing(X_b, alpha))),
        )


class PairingAntisym(Definition):
    """A-π-pairing-antisym: ⟨X_α, β⟩ = −⟨X_β, α⟩."""
    name = "⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)"
    def matches(self, expr):
        return (isinstance(expr, Pairing) and expr.alpha == X_a and expr.X == beta)
    def rewrite(self, expr):
        return Neg(pairing(X_b, alpha))


print("- [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩")
print("- [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩")
print("- ⟨X_α, β⟩ = −⟨X_β, α⟩  (π anti-symmetry)")

- [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
- [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
- ⟨X_α, β⟩ = −⟨X_β, α⟩  (π anti-symmetry)


## 3. Engine

`default_engine` already carries $d^2=0$, pairing rules, and product-rule canonical form. We add the three problem axioms on top.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(KoszulDef_ab())
engine.register(KoszulDef_ba())
engine.register(PairingAntisym())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 11 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
 - [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
 - ⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)


## 4. Goal and proof

$$
[\alpha, \beta]_K \;=\; -[\beta, \alpha]_K.
$$

In [5]:
lhs = koszul_ab
rhs = Neg(koszul_ba)

print(f"LHS: {lhs}")
print(f"RHS: {rhs}")

chain = ExpandAndSimplify().prove(lhs, rhs, registry=reg, engine=engine)
print(f"\nCLOSED, {len(chain)} steps.")

LHS: [α,β]_K
RHS: (-[β,α]_K)

CLOSED, 5 steps.


## 5. Proof chain, LaTeX

In [6]:
from jacopy.display.jupyter import display_chain
display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
[\alpha,\beta]_K \to L_{X_\alpha}\!\left(\beta\right) - L_{X_\beta}\!\left(\alpha\right) - d\!\left(\langle X_\alpha,\, \beta \rangle\right) \quad \text{[[\ensuremath{\alpha},\ensuremath{\beta}]\_K = L\_X\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_X\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle}]\,(axiom)} \\
\langle X_\alpha,\, \beta \rangle \to -\langle X_\beta,\, \alpha \rangle \quad \text{[\ensuremath{\langle}X\_\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} = \ensuremath{-}\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle} (\ensuremath{\pi} anti-symmetry)]\,(axiom)} \\
[\beta,\alpha]_K \to L_{X_\beta}\!\left(\alpha\right) - L_{X_\alpha}\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right) \quad \text{[[\ensuremath{\beta},\ensuremath{\alpha}]\_K = L\_X\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} L\_X\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} d\ensuremath{\langle}X\_\ensuremath{\beta}, \ensuremath{\alpha}\ensuremath{\rangle}]\,(axiom)} \\
\left(L_{X_\alpha}\!\left(\beta\right) - L_{X_\beta}\!\left(\alpha\right) - d\!\left(-\langle X_\beta,\, \alpha \rangle\right)\right) - \left(-\left(L_{X_\beta}\!\left(\alpha\right) - L_{X_\alpha}\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) \to \left(L_{X_\alpha}\!\left(\beta\right) - L_{X_\beta}\!\left(\alpha\right) - \left(-d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) - \left(-\left(L_{X_\beta}\!\left(\alpha\right) - L_{X_\alpha}\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(L_{X_\alpha}\!\left(\beta\right) - L_{X_\beta}\!\left(\alpha\right) - \left(-d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) - \left(-\left(L_{X_\beta}\!\left(\alpha\right) - L_{X_\alpha}\!\left(\beta\right) - d\!\left(\langle X_\beta,\, \alpha \rangle\right)\right)\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{gather*}
}

## 6. Step by step

The flow of the chain (5 steps):

1. **A-K1**, LHS expansion: $[\alpha,\beta]_K \to L_{X_\alpha}\beta - L_{X_\beta}\alpha - d\langle X_\alpha,\beta\rangle$.
2. **A-π-pairing-antisym**, $\langle X_\alpha,\beta\rangle \to -\langle X_\beta,\alpha\rangle$. The single geometric step.
3. **A-K2**, expansion of $[\beta,\alpha]_K$ on the RHS (under negation).
4. **product-rule**, canonical-form pipeline for `Neg`s: double-negation cancellation, `Sum` distribute.
5. **simplify**, canonical form: all $L_X$ terms and pairing $d$'s pair up and drop to $0$.

In [7]:
for i, step in enumerate(chain.steps, 1):
    print(f"[{i}] {step.rule}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] [α,β]_K = L_Xα(β) − L_Xβ(α) − d⟨X_α, β⟩
    [α,β]_K
 -> (L_X_α(β) + (-L_X_β(α)) + (-d(⟨X_α, β⟩)))

[2] ⟨X_α, β⟩ = −⟨X_β, α⟩ (π anti-symmetry)
    ⟨X_α, β⟩
 -> (-⟨X_β, α⟩)

[3] [β,α]_K = L_Xβ(α) − L_Xα(β) − d⟨X_β, α⟩
    [β,α]_K
 -> (L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩)))

[4] product-rule
    ((L_X_α(β) + (-L_X_β(α)) + (-d((-⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))
 -> ((L_X_α(β) + (-L_X_β(α)) + (-(-d(⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))

[5] simplify
    ((L_X_α(β) + (-L_X_β(α)) + (-(-d(⟨X_β, α⟩)))) + (-(-(L_X_β(α) + (-L_X_α(β)) + (-d(⟨X_β, α⟩))))))
 -> 0



## Conclusion

$$
\boxed{\;[\alpha, \beta]_{T^*M} = -[\beta, \alpha]_{T^*M}.\;}
$$

**5-step chain, 3 inline axioms.** Comparison with 2d:

| Pass | Steps | Axioms | Geometric content |
|---|---|---|---|
| 2d (Leibniz) | 13 | 4 | sharp linearity + $L_{fX}$ + pairing linearity + π anti-sym |
| 2e (anti-sym) | 5 | 3 | π anti-sym only |

**Phase 12 tags.** Ideally:

- **Infrastructure #11**, `AntiSymmetric` registry property + bivector evaluation rewrite ($\pi(\alpha,\beta)\to-\pi(\beta,\alpha)$): A-π-pairing-antisym drops from axiom to theorem.
- **12.C(f) `KoszulProblem(π, forms, engine)` wrapper**: registers A-K1 + A-K2 automatically.

When these two land together, the 2e callsite drops to **zero axioms**, only the pure engine chain remains.